In [1]:
# ============================================================
# NB10_Regime_Allocator
# Dynamically allocates portfolio weights based on market regime
# Output: gold_portfolio_allocation
# ============================================================
import uuid
import pandas as pd
from pyspark.sql import functions as F
from datetime import datetime

RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

# 1. Get latest market regime
regime = (
    spark.table("silver_market_regime")
    .orderBy(F.col("trade_date").desc())
    .limit(1)
    .select("market_regime", "trade_date", "close", "sma_200")
)

regime_df = regime.toPandas()

if len(regime_df) == 0:
    print("⚠️ No market regime data found")
    regime_status = "UNKNOWN"
else:
    row = regime_df.iloc[0]
    regime_status = row["market_regime"]
    nifty_close = row["close"]
    nifty_sma = row["sma_200"]
    pct_from_sma = ((nifty_close / nifty_sma) - 1) * 100

print(f"Market Regime: {regime_status}")
print(f"Nifty: {nifty_close:.2f}, SMA200: {nifty_sma:.2f}, {pct_from_sma:+.1f}% from SMA")

# 2. Define allocation rules
if regime_status == "RISK_ON":
    allocation = {
        "GRAHAM": 0.50,
        "GREENBLATT": 0.20,
        "ONEIL": 0.30,
        "LYNCH": 0.00,
        "BUFFETT": 0.00,
    }
    regime_label = "Bull Market — Growth favored"
elif regime_status == "RISK_OFF":
    allocation = {
        "GRAHAM": 0.70,
        "GREENBLATT": 0.20,
        "ONEIL": 0.00,
        "LYNCH": 0.00,
        "BUFFETT": 0.10,
    }
    regime_label = "Bear Market — Defensive favored"
else:
    allocation = {
        "GRAHAM": 0.60,
        "GREENBLATT": 0.20,
        "ONEIL": 0.10,
        "LYNCH": 0.00,
        "BUFFETT": 0.10,
    }
    regime_label = "Uncertain — Balanced allocation"

# 3. Get latest candidates per style
candidates = (
    spark.table("gold_stock_candidates")
    .filter(F.col("screen_date") == F.expr("(SELECT MAX(screen_date) FROM gold_stock_candidates)"))
    .toPandas()
)

# 4. Build allocation table
alloc_rows = []
for style, weight in allocation.items():
    style_picks = candidates[candidates["symbol"].isin(
        candidates[candidates["research_status"].str.contains(style.upper() if style != "ONEIL" else "GROWTH", case=False)]["symbol"]
    )] if len(candidates) > 0 else pd.DataFrame()
    
    alloc_rows.append({
        "style": style,
        "target_weight": weight,
        "num_picks": len(style_picks),
        "weight_per_pick": round(weight / len(style_picks), 4) if len(style_picks) > 0 else 0,
        "regime": regime_status,
        "regime_label": regime_label,
        "allocation_date": datetime.utcnow().date(),
        "run_id": RUN_ID
    })

alloc_df = pd.DataFrame(alloc_rows)

# 5. Save to Lakehouse
spark.createDataFrame(alloc_df).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_portfolio_allocation")

print(f"\n✅ Allocation saved: {len(alloc_df)} styles")
display(alloc_df)

StatementMeta(, 7a92dd44-0351-4328-bae6-444b3987ac99, 3, Finished, Available, Finished, False)

Market Regime: RISK_OFF
Nifty: 23346.40, SMA200: 24494.26, -4.7% from SMA

✅ Allocation saved: 5 styles


SynapseWidget(Synapse.DataFrame, 5bdbc12c-9743-447d-aaf3-c3aa9ef670ad)